<pre>
#+TITLE: Project 2: Analyzing the 2008 Financial Crisis
#+AUTHOR: Edgar Huamantla
#+DATE: 2026-03-22
</pre>

In [ ]:
# Because jupyter notebook has a global name space; use this area for imports
import pandas as pd
import numpy as np
import plotly.express as px
import math
from typing import Union

## SQLAlchemy imports
from sqlalchemy import create_engine
from sqlalchemy.types import Date, Float, String, Integer, Boolean

# aux packages
from IPython.display import Image

# Obtain resource files

In [ ]:
import pathlib
import importlib.util

#check if gdown is installed in current python environment; alert the user to activate their virtual environment to avoid installing to global
if importlib.util.find_spec("gdown") is None:
    print("gdown is not installed. Please activate your virtual environment and install gdown to proceed.")

# dictionary: filename and gdown id
file_names = {'USE3L0712.RSK': '1Y0WE4cqbZfdNEvFuWIDR_d6oDbHHNkly',
              'USE3L0812.RSK': '1Z3vdd5m8uoB4whomq92DQbZHJiKYJc0v'}

curr_working_dir = pathlib.Path.cwd()
print(f"Current working directory: {curr_working_dir}")

for file_name, gdown_id in file_names.items():
    file_path = curr_working_dir / file_name
    if not file_path.exists():
        print(f"File {file_path} does not exist.")
        !gdown {gdown_id} -O {file_path}
    else:
        print(f"File {file_path} already exists. No need to download.")




In [ ]:
# check file integrity of downloaded files
for file_count, file_name in enumerate(file_names.keys(), start=1):
    print(f"\nChecking integrity of file {file_count}: {file_name}...")
    ! head -n 5 {file_name}

# Extract 
 - parse text file into panadas file
 - ignore first row with date ; seen in the head output
    - set second row to header

In [ ]:
barra_df_07 = pd.read_csv('USE3L0712.RSK', skiprows=1, header=0)
print(f"\nDataframe shape: {barra_df_07.shape}")
print("head")
display(barra_df_07.head(3))
print("tail")
display(barra_df_07.tail(3))

In [ ]:
# print DataFrame information showing index dtype and columns, non-NA values and memory usage.
barra_df_07.info()

In [ ]:
# show column names
barra_df_07.columns

In [ ]:
# clean column names; first remove leading and trailing white spaces
barra_df_07.columns = barra_df_07.columns.str.strip()

# check if the column name has invalid characters
invalid_characters = {"%": "_pct"}

# check if column begins with a number; if so, prepend with "col_"
def check_name_leads_with_number(column_name):
    if column_name[0].isdigit():
        print(f"Column name '{column_name}' begins with a number. Prepending with 'col_'.")
        return "col_" + column_name
    else:
        return column_name

# clean column names; replace invalid characters and check if column name begins with a number
for invalid_char, replacement in invalid_characters.items():
    barra_df_07.columns = barra_df_07.columns.str.replace(invalid_char, replacement, regex=False)
barra_df_07.columns = barra_df_07.columns.map(check_name_leads_with_number)

# make lowercase
barra_df_07.columns = barra_df_07.columns.str.lower()


In [ ]:
# new column names
barra_df_07.columns

In [ ]:
# Descriptive statistics include those that summarize the central tendency, dispersion and shape of a dataset’s distribution, excluding NaN values.
barra_df_07.describe()

In [ ]:
# box plot numeric columns; find them first

def find_numeric_columns(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"found {len(numeric_cols)} numeric columns in the DataFrame.")
    print(f"Numeric columns: {numeric_cols}")
    
    return numeric_cols

numeric_columns = find_numeric_columns(barra_df_07)

# to fix the the stack of boxplots we define rows
num_rows = math.ceil(len(numeric_columns) / 4) # 4 columns per row


# melt the df; so that it is not on the same axis
df_melted = barra_df_07.melt(
                            value_vars=numeric_columns,
                            var_name="factor_type", 
                            value_name="factor_value"
                            )


fig = px.box(
    df_melted,
    y="factor_value", # numeric column to plot
    facet_col="factor_type", # categorical column to facet by
    facet_col_wrap=4, # like a line break 
    height = 300 * num_rows, # adjust height based on number of rows needed
    template='plotly_dark',
    points='outliers', # show outliers as points
    facet_col_spacing=0.02, # reduce spacing between facets
    facet_row_spacing=0.02, # reduce spacing between rows of facets
)

fig.update_yaxes(matches=None, showticklabels=True) # allow y axes to have different scales



#clean title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))


#fig.show()

# Transform (Data Cleaning)
 - column names - trim leading and trailing white spaces -- done earlier
    - (optional): set make column names lowercase
 - use a dictionary to replace invalid characters and the value with its replacement (i.e. {% : "_pct"})
 - identify how NaN values are encoded (from box plot hbta has -999)
 - set index

In [ ]:
barra_df_07.head(3)

In [ ]:
# display current indices
barra_df_07.index

In [ ]:
barra_df_07.set_index('ticker').index # by itself does not persist, need to assign back to df or use inplace=True
barra_df_07 = barra_df_07.set_index('ticker') # assign back to df to persist the change

In [ ]:
# find non numbers by column
# numeric_columns already holds numeric columns
non_numeric_columns = [col for col in barra_df_07.columns if col not in numeric_columns]
non_numeric_columns

#optionally: barra_df_07.select_dtypes('object')

In [ ]:
# barra_df_07.loc['AAPL'] error
# barra_df_07[non_numeric_columns].str.strip() # does not work because str returns a series. need to apply to columns separately

def strip_value(value: pd.Series) -> str:
    return value.str.strip()

barra_df_07[non_numeric_columns] = barra_df_07[non_numeric_columns].apply(strip_value)

# lambda way
# barra_df_07[non_numeric_columns] = barra_df_07[non_numeric_columns].apply(lambda x: x.str.strip())

In [ ]:
barra_df_07.head(3)

In [ ]:
print(barra_df_07.index[:5])
print(barra_df_07.index.name)

In [ ]:
# apply strip function to index
barra_df_07.index = barra_df_07.index.str.strip()
barra_df_07.index[:5]

### Troubleshooting key error
```python
# look at first/last labels of the index
print(df.index[:5])
# check if ticket is the index
print(df.index.name)
```

```text
Index(['IX    ', 'SAOL  ', 'IXYS  ', 'CDGT  ', 'CVRG  '], dtype='object', name='ticker')
ticker
```

white spaces in ticker

confirm with substring search
```python
aapl_filter = df[df.index.str.contains('AAPL', case=False, na=False)]
aapl_filter.index[:1]
```

```text
#STDOUT
Index(['AAPL  '], dtype='object', name='ticker')
```

In [ ]:
aapl_filter = barra_df_07[barra_df_07.index.str.contains('AAPL', case=False, na=False)]
aapl_filter.index[:1]

In [ ]:
barra_df_07.loc['AAPL']

## look for null values
- -999 is being used to encode null values
    - search in the numeric columns, return the column

In [ ]:
# intially was going to search all columns, but -999 is numeric so we can just search numeric columns for -999
cols_with_nulls = barra_df_07.columns[(barra_df_07 == -999).any()]
cols_with_nulls

In [ ]:
# null counts
null_counts = (barra_df_07['hbta'] == -999).sum()
null_counts

In [ ]:
# replace the -999 with np.nan
barra_df_07['hbta'] = barra_df_07['hbta'].replace(-999, np.nan)

In [ ]:
# earlier we saw the hbta box plot with the -999 values
fig = px.box(
    barra_df_07['hbta'],
    y='hbta',
    points='outliers',
    notched=True,
    template='plotly_dark',
    title='Box plot of hbta after replacing -999 with NaN'
)
fig.show()

# SQL Books/References 
- SQL Queries for Mere Mortals (John Viescas) 
- SQL in a Nutshell, 4th Ed
- Using SQLite

# LOAD
- load into SQL

## Understanding SQL
A database exists to collect and store data in some organized manner for a specific purpose.
Generally, there are two types of databases:
- operational : used to collect, modify and maintain data on a day to day basis.
- analytical : stores and tracks historical and time dependent data. 
    - Data tends to be static -- meaning it is never/rarely modified. New data is often added.

The Father of the relational model, is Dr. Edgar F. Codd. A relation is a table in set theory and the Relational model is based on 
two branches of mathematics:
- 1. Set theory
- 2. First-order predicate logic -- Filters

The father of data warehousing was William (Bill) H. Immon who developed the idea that organizations would access data stored in any number of non-relational databases.

## Anatomy of a Relational Database
Data in a relational database is stored in, you guesssed it, relations. Relations are tables composed of tuples and attributes. 
- Tuples are another way of saying records or rows. 
- Attributes are the fields and columns in a table.
    - Columns is a structure that stores data. It represents a characteristic of the subject of the table.

A table should always represent a single specific subject.
Tables can represent:
- an object: tangible; (think noun) -- person, place or thing.
- event: occurs at a given point in time aand has characteristics you wish to record.

We retrieve data from columns, and if they are properly designed, it only contains only one value.
Rows represent a unique instance of the subject of a table.
- a row is composed of the entire set of columns in a table.

Keys: one or more columns that uniquely identify each row within a table.
- A primary key is important for two reasons, in addition to enforcing table-level integrity and helps establish relationships with
other data.
    - 1. its value represents a specific row throughout the entire database
    - 2. it's column identifies a given table throughout the entire database.

Foreign Keys: take a copy of the primary key form the first table and insert into a second table -- becoming a foreign key. They help to ensure relationship level integrity. Rows in both tables will always be properly related because the value of a foreign key must be drawn from the values of the primary key to which it refers to.
- the second table already has a primary key of its own, and the primary key you are introducing from the first table is foreign to the second table.

Views: are virtual tables composed of columns from one or more tables in the databse. 
- views can be thought of a saved query because only the structure is stored.

### Relationships
- one-to-one: a table has been split into two parts.
- one-to-many: the 'one' has a relationship with many rows in the "many" side.
- many-to-many: these require a 'linking table'. Linking tables are a way of associating rows from one table with those of th oether. The advantage is that it allows you to associate any number of rows from both tables in the relationship.

There is a difference between:
- Database theory: rules and theory that form the basis of the relational model
- Database design: structured, organized set of processes used to design a relational database. It works to ensure integrity, consistency, and accuracy of the data.

# SQLALCHEMY

```python
pip install sqlalchemy
```
SQLAlchemy library is divided into two modules: CORE and ORM (Object Relational Mapping).

Core: contains the database integration logic for supported database dialects, collection of classes to describe database tables, and a system for generating SQL statements using Python language constructs.

ORM: an abstraction layer between Python and the database that allows database operations to be derived from actions performed on Python objects.

## The Database Engine:
SQLAlchemy uses 'engine' objects to manage connections to a database. The create_engine() function constructs an engine given a database URL.

## SQLITE
SQLite is a C-language library that implements a self-contained relational database.

In [ ]:
def get_schema(df: pd.DataFrame) -> dict:
    """
    loop through a dataframe looking at each column and get its dtype name and length
    """
    schema_dictonary = dict()
    for col in df.columns:
        dtype_name = df[col].dtype.name

        # create a nest dictionary for each column with dtype and length
        column_info = {"dtype": dtype_name}
        
        if dtype_name == 'object':
            # get the max length of strings in the column
            max_len = df[col].astype(str).str.len().max()
            column_info["max_length"] = int(max_len) if not pd.isna(max_len) else 0

        elif dtype_name in ['int64', 'float64']:
            unique_values = set(df[col].dropna().unique())
            if unique_values.issubset({0, 1}):
                column_info["dtype"] = 'boolean'

        schema_dictonary[col] = column_info
    return schema_dictonary
            

In [ ]:
type_dict = get_schema(barra_df_07)
type_dict

In [ ]:
# loop through type dictionary and show string columns with their max length
for col, info in type_dict.items():
    if info['dtype'] == 'object':
        print(f'{col} has max length of {info["max_length"]}')

In [ ]:
# String Definitions, adding 50% buffer and rounding up with math.ceil
SQL_TYPE_MAPPING = {
    'int64': Integer,
    'float64': Float,
}

def generate_sqlalchemy_schema(meta_data : dict) -> dict:
    """
    given a dictionary with col as keys and a nested dict with dtype and max length,
    we build a new dictionary that adds a 50% buffer to string lengths,
    if it is int64 or float64 we use a SQL_TYPE_MAPPING constants to use sqlalchemy types,
    we previously labeled columsn with values between 0 and 1 as boolean, so this handles it.   
    
    """
    
    orm_schema = dict()

    for col, info in meta_data.items():
        dtype = info['dtype']
        
        match dtype:
            case 'object':
                max_length = info['max_length']
                buffer_length = math.ceil(max_length * 1.5) # add 20% buffer and round up
                orm_schema[col] = String(buffer_length)
            case 'int64' | 'float64':
                orm_schema[col] = SQL_TYPE_MAPPING[dtype]
            case 'boolean':
                orm_schema[col] = Boolean
            case _:
                print(f"Data type {dtype} for column {col} not recognized.")
                
    return orm_schema


In [ ]:
dtype_dict = generate_sqlalchemy_schema(type_dict)
dtype_dict

### NOTE ON CONNECTION ENGINE
SQLITE:
- 3 slashes is relative paths
- 4 slashes is for absolute paths <br>
reference: https://docs.sqlalchemy.org/en/21/core/engines.html#sqlite

In [ ]:
# load dataframe to SQL and keep index because we set it in the df
# wrap it around an if statement to check for file existence;
# if file exists skip

db_name = 'barra_database.db'
engine = create_engine(f'sqlite:///{db_name}')
db_path = curr_working_dir / db_name
if not db_path.exists():
    print(f"Database {db_path} does not exist. Loading DataFrame to SQL.")
    barra_df_07.to_sql(
        name='barra_factors',
        con=engine,
        if_exists='fail',
        index=True,
        dtype=dtype_dict
    )
else:
    print(f"Database {db_path} already exists. Skipping loading DataFrame to SQL.")

# SQL Queries for Barra Factor Analysis

In [ ]:
image_folder = curr_working_dir / 'src' / 'images'
Image(image_folder / 'SELECT_STATEMENT_mere-mortals.jpg', width=600)

# SQL Queries for Barra Factor Analysis

## 1. Basic Count Query
```sql
SELECT COUNT(*) as total_records
FROM barra_factors;
```
This query gives us the total number of records in our database. It's useful for verifying that all data was loaded correctly.

TRANSLATION: COUNT all rows, name it total_records, FROM barra_factors


In [ ]:
count_query = """
SELECT
    COUNT(*) as total_records
FROM barra_factors;
"""

In [ ]:
db_records = pd.read_sql(count_query, engine)
print(f'Total records:\n {db_records}')

In [ ]:
# inspect the struc of db_records
db_records
print(f'type of db_records: {type(db_records)}')

In [ ]:
# assert that database records match dataframe records
assert len(barra_df_07) == db_records['total_records'][0]
print(f'barra_df_07 records: {len(barra_df_07)}, db_records: {db_records["total_records"][0]}. Assertion passed, record counts match.')


## 2. Summary Statistics Query
```sql
SELECT
    COUNT(*) as count,
    AVG(BETA) as avg_beta,
    AVG(SIZE) as avg_size,
    AVG(MOMENTUM) as avg_momentum,
    AVG(VALUE) as avg_value
FROM barra_factors;
```
This query calculates average values for key Barra factors. Common aggregate functions include:
- `AVG()`: Average value
- `STDDEV()`: Standard deviation
- `MIN()`: Minimum value
- `MAX()`: Maximum value


In [ ]:
Image(image_folder / 'AGG_FUNCS_mere-mortals.jpg', width=600)

In [ ]:
stats_query = '''
SELECT
    COUNT(*) as count,
    AVG(beta) as avg_beta,
    AVG(size) as avg_size,
    AVG(momentum) as avg_momentum,
    AVG(value) as avg_value
FROM barra_factors;    
'''
stats_results = pd.read_sql(stats_query, engine)
print(f'Summary statistics from SQL query:\n {stats_results}')

In [ ]:
Image(image_folder / 'GROUP_BY-mere-mortals.jpg', width=600)

# GROUP BY 

```sql
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
```
This query demonstrates:
- `GROUP BY`: Aggregating data by category
- `ORDER BY`: Sorting results
- `LIMIT`: Restricting number of results


GROUP BY
* **Collapses** rows into summary statistics
* Returns one row per group
* Loses individual row details


## Grouping Data
GROUP BY forms subsets of rows based on matching values. It allows you to choose the columns from a table result letting the database use them as definitions for how to group the rows.
- Rows that have the same value in the list of chosen columns are collected into a group. 
- GROUP BY cannot be applied to an expression (a logical statement that evaluates to true/false/unknown)

In [ ]:
group_by_indname_query_by_count = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS count,
    AVG(capitalization) AS avg_capitalization
FROM barra_factors
GROUP BY indname1
ORDER BY count DESC
LIMIT 5;
'''
group_by_count_results = pd.read_sql(group_by_indname_query_by_count, engine)
print(f'Top 5 Industry Distributions by count:\n {group_by_count_results}')

In [ ]:
group_by_indname_by_cap_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS count,
    AVG(capitalization) AS avg_capitalization
FROM barra_factors
GROUP BY indname1
ORDER BY avg_capitalization DESC
LIMIT 5;
'''
group_by_cap_results = pd.read_sql(group_by_indname_by_cap_query, engine)
print(f'Top 5 Industry Distributions by average capitalization:\n {group_by_cap_results}')

## Window Functions

* **Maintains** all rows
* Adds calculations to each row
* Can see individual values alongside aggregates

Window functions allows for working with data and its adjacent rows allowing for operations such as generating running totals.
A window function:
- maps a vector to a function and produces a vector of the same length.
    - in contrast, an aggregate function reduces a vector to a scalar.
- aggregated data is replaced by sub-total rows preserving the details of the data that went into the aggregation functions

OVER can be thought as a context manager that defines the scope (the data) the function can see. OVER applies a function and broadcast it across a given vector of rows.
Context managers exist to provide a safe container for an operation to happen without affecting the rest of the data -- defines the visibilty.
SQL context managers manage the 'data window' solving the problem of an aggregate function being global, meaning it can see everything and collapses into one number. OVER sets up a temporary scope allowing for everything inside this clause to happen under a specific set of rules, once completed, the rules disappear.

WHERE filters down to one group.

In [ ]:
schema_info_query = pd.read_sql("PRAGMA table_info('barra_factors');", engine)

size_query_limit_5 = '''
SELECT
    ticker,
    size
FROM barra_factors
ORDER BY size DESC
LIMIT 5;
'''
print(f'5 stocks with size factor values:\n {pd.read_sql(size_query_limit_5, engine)}')




In [ ]:
# print liquid stocks
liquid_stocks_query_estuni_gt_1 = '''
SELECT
    name
FROM barra_factors
WHERE e3estu = TRUE AND size > 1 --- Liquid US stocks within ESTUNI and size greater than 1
ORDER BY size DESC;
'''
liquid_stocks_query_estuni_gt_1_results = pd.read_sql(liquid_stocks_query_estuni_gt_1, engine)
print(f'Liquid stocks WHERE ESTUNI = True and size > 1:\n {liquid_stocks_query_estuni_gt_1_results}')

In [ ]:
# industry distibution of liquid stocks
industry_dist_liquid_stocks_query = '''
SELECT
    name,
    indname1 AS industry,
    leverage,
    AVG(leverage) OVER (PARTITION by INDNAME1) AS avg_industry_leverage,
    leverage - AVG(leverage) OVER (PARTITION by INDNAME1) as diff_from_avg
FROM barra_factors
WHERE e3estu = TRUE AND size > 1 --- Liquid US stocks within ESTUNI and size greater than 1
ORDER BY size DESC;
'''
industry_dist_liquid_stocks_query_results = pd.read_sql(industry_dist_liquid_stocks_query, engine)
print(f'Industry distribution of liquid stocks WHERE ESTUNI = True and size > 1:\n {industry_dist_liquid_stocks_query_results}')

In [ ]:
# liquid us banks within ESTUNI and size greater than -1
liquid_us_banks_query_estuni_gt_neg1 = '''
SELECT
    name,
    leverage,
    AVG(leverage) OVER (PARTITION by indname1) AS avg_industry_leverage,
    leverage - AVG(leverage) OVER (PARTITION by indname1) as diff_from_avg
FROM barra_factors
WHERE e3estu = TRUE AND size > -1 AND indname1 = 'BANKS' --- Liquid US banks within ESTUNI and size greater than -1
ORDER BY size DESC;
'''
liquid_us_banks_query_estuni_gt_neg1_results = pd.read_sql(liquid_us_banks_query_estuni_gt_neg1, engine)
print(f'Liquid US banks within ESTUNI and size greater than -1:\n {liquid_us_banks_query_estuni_gt_neg1_results}')

# DISTINCT
To get unique industry names (INDNAME1), you can use the DISTINCT keyword in SQL. Here's how:

In [ ]:
# query to get unique industry names
unique_industries_query = '''
SELECT
    DISTINCT indname1 AS industry
FROM barra_factors
ORDER BY industry;
'''
unique_industries_results = pd.read_sql(unique_industries_query, engine)
print(f'Unique industry names:\n {unique_industries_results}')

In [ ]:
# query to show how many unique industries exist
unique_industries_count_query = '''
SELECT
    COUNT(DISTINCT indname1) AS unique_industry_count
FROM barra_factors;
'''
unique_industries_count_results = pd.read_sql(unique_industries_count_query, engine)
print(f'Number of unique industries:\n {unique_industries_count_results}')

# Understanding SQL Filtering with WHERE and AND

Let's explore how to filter data in SQL using a practical example with our Barra dataset. We'll look at how to find specific industries and add additional conditions.

## Basic Industry Query
First, let's look at our base query:
```python
# Basic industry distribution
basic_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(basic_industry_query, engine)
```

## Adding WHERE Clause
Now let's add filtering to focus on large-cap stocks:
```python
# Adding WHERE to filter results
filtered_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
WHERE CAPITALIZATION > 1000000000  -- Filter for large cap stocks > $1bn
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(filtered_industry_query, engine)
```

## Using Multiple Conditions with AND
Let's add more conditions to find large-cap stocks with positive momentum:
```python
# Using WHERE with multiple conditions
complex_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap,
    AVG(MOMENTUM) as avg_momentum
FROM barra_factors
WHERE CAPITALIZATION > 1000000000     -- First condition
    AND MOMENTUM > 0               -- Second condition
    AND NONESTU = 0               -- Third condition
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(complex_industry_query, engine)
```

## Understanding SQL Filtering
1. **WHERE Clause**:
   - Acts like a filter on your data
   - Goes after the FROM statement
   - Only rows that meet the condition are included
   - Example: `WHERE CAPITALIZATION > 1000000` only shows large-cap stocks

2. **AND Operator**:
   - Combines multiple conditions
   - ALL conditions must be true for the row to be included
   - Think of it like a series of filters
   - Example: `WHERE CAPITALIZATION > 1000000 AND MOMENTUM > 0` shows only large-cap stocks with positive momentum

3. **Order of Operations**:
   ```sql
   SELECT columns           -- 3. Select specific columns
   FROM table              -- 1. Start with table
   WHERE conditions        -- 2. Filter rows
   GROUP BY column         -- 4. Group results
   ORDER BY column         -- 5. Sort results
   LIMIT number;           -- 6. Limit number of rows
   ```


In [ ]:
Image(image_folder / 'WHERE-mere-mortals.jpg', width=600)

In [ ]:
Image(image_folder / 'AND_OPERATOR-mere-mortals.jpg', width=600)

In [ ]:
# Basic industry query
basic_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""
basic_industry_query_results = pd.read_sql(basic_industry_query, engine)
print(f'Industry distribution:\n {basic_industry_query_results}')



In [ ]:
# adding WHERE clause
filtered_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
WHERE CAPITALIZATION > 1000000000  -- Filter for large cap stocks > $1bn
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""
filtered_industry_query_results = pd.read_sql(filtered_industry_query, engine)
print(f'Industry distribution for large cap stocks:\n {filtered_industry_query_results}')

In [ ]:
# using multiple conditions with AND
complex_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap,
    AVG(MOMENTUM) as avg_momentum
FROM barra_factors
WHERE CAPITALIZATION > 1000000000     -- First condition
    AND MOMENTUM > 0               -- Second condition
    AND NONESTU = 0               -- Third condition
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""
complex_industry_query_results = pd.read_sql(complex_industry_query, engine)
print(f'Industry distribution for large cap stocks:\n {complex_industry_query_results}')

# Understanding the IN Operator in SQL

## What is the IN Operator?
The IN operator is a powerful SQL feature that allows you to check if a value matches any value in a list of values. It's similar to saying "where x equals this OR that OR that..."

## Basic Syntax
```sql
WHERE column_name IN (value1, value2, value3)
```

## Comparison with OR
Consider these two equivalent queries:

Using OR:
```sql
WHERE INDNAME1 = 'BANKS'
   OR INDNAME1 = 'THRIFTS'
   OR INDNAME1 = 'SECASSET'
```

Using IN:
```sql
WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET')
```

The IN operator is:
- More concise
- Easier to read
- Less prone to errors
- Often more efficient

## Real Example with Barra Data
```python
# Finding all financial companies
financial_query = """
SELECT
    INDNAME1 as Industry,
    COUNT(*) as Count
FROM barra_factors
WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS')
GROUP BY INDNAME1;
"""
```

## Advanced Features
1. **Can use with subqueries:**
```sql
WHERE MOMENTUM IN (SELECT TOP 10 MOMENTUM FROM barra_factors)
```

2. **Can use NOT IN:**
```sql
WHERE INDNAME1 NOT IN ('BANKS', 'THRIFTS')
```

3. **Works with multiple types:**
```sql
WHERE value IN (100, 200, 300)  -- numbers
WHERE status IN ('active', 'pending')  -- strings
```


In [ ]:
Image(image_folder / 'IN-OPERATOR-mere-mortals.jpg', width=600)

In [ ]:
# financial services companies query using Barra Handbook pg 113-114
Image(image_folder / 'barra-handbook-appdx-b_excerpt-fin.jpg', width=400)

In [ ]:
# find financial companies
# search for keywords that are like:
# 'life', 'property', 'bank', 'thrift', 'sec', 'financial', 'lifeins', 'insurance','otherins', 'bank', 'thrift', 'secasset', 'inservbr'
financial_query = """
SELECT
    indname1 as industry,
    COUNT(*) as count
FROM barra_factors
WHERE indname1 in ('BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS') AND nonestu = 0
GROUP BY indname1;
"""
financial_query_results = pd.read_sql(financial_query, engine)
print(f'Financial services industry distribution:\n {financial_query_results}')

In [ ]:
# create list of columns to search for using indname + digit from range(1,6) // 6 is exclusive so it goes up to 5
indname_columns_list = [f'indname{i}' for i in range(1, 6)]
fin_keywords_list = ['life', 'property', 'bank', 'thrift', 'sec', 'financial', 'lifeins', 'insurance','otherins', 'bank', 'thrift', 'secasset', 'inservbr']
conditions = [
    f"{col} LIKE '%{kw}%'"
    for col in indname_columns_list
    for kw in fin_keywords_list
]
print(conditions[:5])

where_clause = " OR ".join(conditions)

financial_query = f"SELECT * FROM barra_factors WHERE {where_clause};"
#print(financial_query)
financial_query_results = pd.read_sql(financial_query, engine)
#print(f'Financial companies search results:\n {financial_query_results.head(3)}')

In [ ]:
# verify which columns i should be searching in
def contains_keyword(val):
    if not isinstance(val, str): return False
    return any(kw in val.lower() for kw in fin_keywords_list)

hit_map = financial_query_results[indname_columns_list].applymap(contains_keyword)
print(f"Hit map showing which columns contained financial keywords:\n {hit_map}")

column_hit_count = hit_map.sum().sort_values(ascending=False)
print(f"Count of hits per column:\n {column_hit_count}")

In [ ]:
# confirmed indname1 is the column to search keywords
target_col = 'indname1'
conditions_indname1 = [f"{target_col} LIKE '%{kw}%'" for kw in fin_keywords_list]
where_clause_indname1 = " OR ".join(conditions_indname1)
financial_query_indname1 = f"SELECT indname1, ticker FROM barra_factors WHERE {where_clause_indname1};"
print(financial_query_indname1)


In [ ]:
fin_industry_count_query = f"""
    SELECT
        indname1 AS industry,
        COUNT(*) AS count
    FROM barra_factors
    WHERE {where_clause_indname1} AND NONESTU = 0 --- Filter for financial companies and exclude non-ESTU stocks
    GROUP BY indname1
    ORDER BY count DESC;  
"""
fin_industry_count_results = pd.read_sql(fin_industry_count_query, engine)
print(f'Financial industry counts:\n {fin_industry_count_results}')

# Understanding UNION in SQL

## What is UNION?
UNION is a SQL operator that combines the results of two or more SELECT statements into a single result set. It's particularly useful when you want to:
- Compare metrics across different groups
- Combine similar data from different categories
- Create summary tables with labeled sections

## Basic Syntax
```sql
SELECT column1, column2 FROM table1
UNION
SELECT column1, column2 FROM table2
```

## Key Rules for UNION
1. Each SELECT statement must have the same number of columns
2. Corresponding columns must have similar data types
3. The column names from the first SELECT statement are used as the column names for the result set

## Practical Example: Comparing Utilities
```sql
SELECT
    'Gas' as Sector,            -- Creates label 'Gas'
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions
FROM barra_factors
WHERE INDNAME1 IN ('GASUTIL')
AND NONESTU = 0
UNION
SELECT
    'Electric' as Sector,       -- Creates label 'Electric'
    AVG(CAPITALIZATION)/1e9 as Avg_MarketCap_Billions
FROM barra_factors
WHERE INDNAME1 IN ('ELECUTIL')
AND NONESTU = 0;
```

In this example:
- First query calculates average market cap for gas utilities
- Second query calculates average market cap for electric utilities
- UNION combines them into one table with labeled sectors
- Both queries must return same number of columns (2 in this case)
- Both queries must have compatible types (text and number)


In [ ]:
Image(image_folder / 'UNION-mere-mortals.jpg', width=600)

## Unions
Union allows you to select and combine rows from two or more sets, it makes a table 'taller' meaning it allows you to stack rows on top each other. For a single table, its best to use WHERE. <br>
In contrast, JOINs make a table wider because you are adding columns. <br>
Requirements:
- Same number of columns
- Each column MUST be comparable in terms of datatype.
    - Column names are irrelevant in UNION because only ordinal position plus datatype matter.

In [ ]:
# example template for sector comparison
comparison_query = '''
SELECT
    'Gas' as sector,
    AVG(capitalization)/1e9 as avg_marketcap_billions
FROM barra_factors
WHERE indname1 in ('GASUTIL') --- ... LIKE LOWER(indname) for more flexible matching
AND nonestu - 0;
'''
comparison_query_results = pd.read_sql(comparison_query, engine)
print(f'Gas Utility sector comparison query results:\n {comparison_query_results}')

In [ ]:
comparison_query = '''
SELECT
    'Electric' as sector,
    AVG(capitalization)/1e9 as avg_marketcap_billions
FROM barra_factors
WHERE indname1 IN ('ELECUTIL')
AND nonestu = 0;
'''
comparison_query_results = pd.read_sql(comparison_query, engine)
print(f'Electric Utility sector comparison query results:\n {comparison_query_results}')

In [ ]:
# example template for sector comparison
comparison_query = '''
SELECT
    'Gas' as sector,
    AVG(capitalization)/1e9 as avg_marketcap_billions
FROM barra_factors
WHERE indname1 in ('GASUTIL')
AND nonestu = 0
UNION
SELECT
    'Electric' as sector,
    AVG(capitalization)/1e9 as avg_marketcap_billions --- dividing by scalar literal to convert to billions
FROM barra_factors
WHERE indname1 IN ('ELECUTIL')
AND nonestu = 0;
'''
comparison_query_results = pd.read_sql(comparison_query, engine)
print(f'Gas vs Electric Utility sector comparison query results:\n {comparison_query_results}')


# Student Research Questions: Analyzing Sectors During the GFC

## Important Note
For all queries, remember to include `AND NONESTU = 0 AND SIZE>0` to restrict analysis to the larger companies within the estimation universe (i.e. no foreign companies). This ensures we're working with actively traded, liquid stocks that Barra uses in their factor model.

## Tips for Analysis
1. Always start queries with checking how many stocks are in each sector (N) in the estimation universe
2. Consider size differences when comparing sectors
3. Look at both averages and distributions of factors
4. Consider combining multiple factors in your analysis
5. Think about appropriate presentation formats (averages, rankings, etc.)
6. Remember to consider market cap effects in all analyses


## Core Research Questions

In [ ]:
SECTORS = {
    'FINANCIAL': ['BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS', 'PRPTYINS', 'LIFEINS'],
    'TECH_COMM': ['SEMICOND', 'CMPTRSW', 'CMPTRHW', 'INTERNET', 'INFOSVCS',
                  'ELECEQP', 'TELEPHON', 'WIRELESS', 'MEDIA', 'PUBLISH'],
    'REALESTATE_UTIL': ['EQTYREIT', 'ELECUTIL', 'GASUTIL'],
    'HEALTHCARE': ['MEDPRODS', 'BIOTECH', 'DRUGS', 'MEDPROVR'],
    'CONS_DISCR': ['SPLTYRET', 'RESTRNTS', 'CLOTHING', 'APPAREL', 'DEPTSTOR',
                   'HOTELS', 'ENTRTAIN', 'LEISURE', 'MOTORVEH', 'CONSDUR'],
    'CONS_STAPLES': ['GROCERY', 'FOODBEV', 'HOMEPROD', 'ALCOHOL', 'TOBACCO'],
    'INDUSTRIAL': ['CONSTRUC', 'INDPART', 'INDSVCS', 'HEAVYELC', 'HEAVYMCH',
                  'DEFAERO', 'ENVSVCS'],
    'BASIC_MAT': ['CHEMICAL', 'FOREST', 'GOLD'],
    'ENERGY': ['ENGYRES', 'OILSVCS', 'OILREF', 'MINING'],
    'TRANSPORT': ['AIRLINES', 'RAILROAD', 'TRUCKFRT']
}

1. **Risk Characteristics**
   ```sql
   -- Always include: AND NONESTU = 0
   SELECT
       AVG(BETA) as avg_beta,
       AVG(VOLTILTY) as avg_volatility
   FROM barra_factors
   WHERE INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS')
   AND NONESTU = 0;   -- Critical for estimation universe
   ```
   * How do betas compare between financial and technology sectors?
   * Which sector has higher volatility?
   * Is systematic risk (SRISK%) significantly different?

In [ ]:
# * How do betas compare between financial and technology sectors?
betas_between_financial_and_tech_query = f'''
SELECT
    indname1 AS industry,
    AVG(beta) as avg_beta
FROM barra_factors
WHERE indname1 IN {tuple(SECTORS['FINANCIAL'])}
AND nonestu = 0
GROUP BY indname1
UNION
SELECT
    indname1 AS industry,
    AVG(beta) as avg_beta
FROM barra_factors
WHERE indname1 IN {tuple(SECTORS['TECH_COMM'])}
AND nonestu = 0
GROUP BY indname1
ORDER BY avg_beta DESC;
'''
betas_between_financial_and_tech_results = pd.read_sql(betas_between_financial_and_tech_query, engine)
print(f'Average beta between financial and tech sectors:\n {betas_between_financial_and_tech_results}')

In [ ]:
# * Which sector has higher volatility?
vol_bw_financial_and_tech_query = f'''
SELECT
    indname1 AS industry,
    AVG(voltilty) as avg_volatility
FROM barra_factors
WHERE indname1 in {tuple(SECTORS['FINANCIAL'])} OR indname1 in {tuple(SECTORS['TECH_COMM'])}
AND nonestu = 0
GROUP BY indname1
ORDER BY avg_volatility DESC;
'''
vol_bw_financial_and_tech_results = pd.read_sql(vol_bw_financial_and_tech_query, engine)
print(f'Average volatility between financial and tech sectors:\n {vol_bw_financial_and_tech_results}')

In [ ]:
# search the dict_type mapping from earlier
keyword = 'srisk'
match_kw = [col for col in dtype_dict.keys() if keyword in col.lower()]
print(f"Columns matching keyword '{keyword}': {match_kw}")

In [ ]:
# * Is systematic risk (SRISK%) significantly different?
srisk_bw_financial_and_tech_query = f'''
SELECT
    indname1 AS industry,
    AVG(srisk_pct) as avg_srisk_pct
FROM barra_factors
WHERE indname1 in {tuple(SECTORS['FINANCIAL'])} OR indname1 in {tuple(SECTORS['TECH_COMM'])}
AND nonestu = 0
GROUP BY indname1
ORDER BY avg_srisk_pct DESC;
'''
srisk_bw_financial_and_tech_results = pd.read_sql(srisk_bw_financial_and_tech_query, engine)
print(f'Average SRISK% between financial and tech sectors:\n {srisk_bw_financial_and_tech_results}')

2. **Size and Trading**
   * Compare market capitalization between two sectors (remember NONESTU = 0)
   * Is trading activity (TRADEACT) different?
   * Do certain sectors show more momentum?

In [ ]:
# * Compare market capitalization between two sectors (remember NONESTU = 0)
comp_market_cap_bw_banks_and_lifeins_query = '''
SELECT
    indname1 AS industry,
    AVG(capitalization)/1e9 as avg_marketcap_billions
FROM barra_factors
WHERE indname1 IN ('BANKS', 'LIFEINS') AND nonestu = 0
GROUP BY indname1
ORDER BY avg_marketcap_billions DESC;
'''
comp_market_cap_bw_banks_and_lifeins_results = pd.read_sql(comp_market_cap_bw_banks_and_lifeins_query, engine)
print(f'Average market cap between banks and life insurance sectors:\n {comp_market_cap_bw_banks_and_lifeins_results}') 

In [ ]:
# search for tradeact
keyword = 'trade'
match_kw = [col for col in dtype_dict.keys() if keyword in col.lower()]
print(f"Columns matching keyword '{keyword}': {match_kw}")

In [ ]:
# * Is trading activity (TRADEACT) different?
comp_market_cap_bw_banks_and_lifeins_w_tradeact_query = '''
SELECT
    CASE
        WHEN indname1 IN ('BANKS') THEN 'BANKS'
        WHEN indname1 IN ('LIFEINS') THEN 'LIFEINS'
        ELSE 'ALL_OTHERS'
    END AS industry,
    AVG(tradeact) as avg_tradeact
FROM barra_factors
WHERE nonestu = 0
GROUP BY industry
ORDER BY avg_tradeact DESC;
'''
comp_market_cap_bw_banks_and_lifeins_w_tradeact_results = pd.read_sql(comp_market_cap_bw_banks_and_lifeins_w_tradeact_query, engine)
print(f'Average market cap and trade activity between banks and life insurance sectors:\n {comp_market_cap_bw_banks_and_lifeins_w_tradeact_results}')

In [ ]:
#comp_market_cap_bw_banks_and_lifeins_results.info()
bank_trade_activity = comp_market_cap_bw_banks_and_lifeins_w_tradeact_results[comp_market_cap_bw_banks_and_lifeins_w_tradeact_results['industry'] == 'BANKS']['avg_tradeact']
life_ins_trade_activity = comp_market_cap_bw_banks_and_lifeins_w_tradeact_results[comp_market_cap_bw_banks_and_lifeins_w_tradeact_results['industry'] == 'LIFEINS']['avg_tradeact']
print(f"Difference in average trading activity between banks and life insurance sectors:\n {bank_trade_activity.iloc[0] - life_ins_trade_activity.iloc[0]:.4f}")

Is trading activity (TRADEACT) different? <br>
Trading activity is similar between bank and Lifeinsurance, they are only slightly different by 2.6 percentage points. When compared to other sectors (excluding Life Insurance and Banks) we see a positive average trading activity of +.0982

## Advanced Research Questions

In [ ]:
# # search the dict_type mapping from earlier
# keyword = 'srisk'
# match_kw = [col for col in dtype_dict.keys() if keyword in col.lower()]
# print(f"Columns matching keyword '{keyword}': {match_kw}")

def search_kw(keyword):
    keyword = keyword
    matched_kw = [col for col in dtype_dict.keys() if keyword in col.lower()]
    return matched_kw

In [ ]:
lev_kws = search_kw('lev')
lev_kws

3. **Leverage Analysis**
   ### Question 3.1: High Leverage Financial Institutions
   * Find top 10 banks by leverage ratio
   * Compare their market caps and volatility
   * Analyze relationship between leverage and other risk metrics

In [ ]:
earnings_kw = search_kw('earn')
earnings_kw

In [ ]:
top_ten_banks_by_lev_query = '''
SELECT
    name,
    leverage,
    voltilty,
    earnyld as earnings_yield,
    earnvar as earnings_variability,
    capitalization/1e9 as marketcap_bn --- 1B = 1000M = 1e9
FROM barra_factors
WHERE indname1 IN ('BANKS')
AND nonestu = 0
ORDER BY leverage DESC
LIMIT 10;
'''

top_ten_banks_by_lev_results = pd.read_sql(top_ten_banks_by_lev_query, engine)
print(f'Top 10 banks by leverage:\n {top_ten_banks_by_lev_results}')

In [ ]:
# build a correlation matrix of risk metrics queried
corr_matrix = top_ten_banks_by_lev_results.drop(columns=['name']).corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale='RdBu_r', # red-blue diverging color scale
    aspect="auto",
    title='Corr Matrix: Top 10 Leveraged Banks 2007'
)

fig.update_layout(
    xaxis_title='Barra Risk Factors',
    yaxis_title='Barra Risk Factors',
    template='plotly_dark',
    width=600,
    height=600,
)


### Analysis 3.1: High Leverage Financial Institutions 
From definitions found in Barra PDF, page 74. <br>
What stood out between the top 10 was NATIONAL CITY CORP and COLONIAL BANCGROUP INC, each having 1.366 and 1.363, respectively, suggesting that they should behave in a simlar fashion with respect to their returns.<br>
However, we see that their earnings variance, is significantly different between NATIONAL CITY CORP and COLONIAL BANCGROUP INC with 0.847 and 0.27 respectively. National City should have a much higher yield to compensate for the extra fluctuations in earnings but we don't see that. The correlation of 0.17 between the two risk factors, earnings yield and earnings variability, suggest that the market hadn't priced in this fluctuation.

In [ ]:
# compare leverage to volality and show the size of the stock by market cap
df = top_ten_banks_by_lev_results

fig = px.scatter(
    df,
    x='leverage',
    y='voltilty',
    size='marketcap_bn',
    color='earnings_yield',
    hover_name='name',
    text='name',
    size_max=60,
    title='Leverage vs Volatility for Top 10 Leveraged Banks (2007)',
    labels={
        'leverage': 'Leverage',
        'voltilty': 'Volatility',
        'marketcap_bn': 'Market Cap (Billions)',
        'earnings_yield': 'Earnings Yield'
    },
)


# clean up label and positioning and style
fig.update_traces(textposition='top center')
fig.update_layout(
    template='plotly_dark',
    height=700,
    xaxis=dict(showgrid=True, zeroline=False),
    yaxis=dict(showgrid=True, zeroline=False)
)
fig.show()

### Analysis 3.1: High Leverage Financial Institutions 
When we add size of a bank to our analysis, we see that leverage isn't displaying a relationship with volality as leverage grows. In fact, we see that that the larger bank, by market capitalization, has a more stable price when compared to other banks offering the same yield and similar leverage like CENTRAL PAC. We also see that National City Corp, a larger bank with more leverage, has similar earnings yield to Colonial Bank despite having higher volatlity. What stands out is First Horizon being more leveraged and having higher volatility but having a lower earning yield. Despite Popular being more leverage, and lower volatility we see its paying a similar yield than First Horizon National Corp, suggesting that something else is driving volatility.

### Question 3.2: Industry Leverage Comparison
   * Rank industries by average leverage
   * Compare financial subsectors
   * Consider size effects in leverage analysis

In [ ]:
industry_by_avg_lev_query = '''
SELECT
    indname1 AS industry,
    AVG(leverage) AS avg_leverage
FROM barra_factors
WHERE nonestu = 0
GROUP BY indname1
ORDER BY avg_leverage DESC;
'''
industry_by_avg_lev_results = pd.read_sql(industry_by_avg_lev_query, engine)
print(f'Average leverage by industry:\n {industry_by_avg_lev_results}')

In [ ]:
sub_sector_avg_lev_marketcap_query = f" \
SELECT \
    indname1, \
    AVG(leverage) as avg_leverage, \
    capitalization/1e9 as marketcap_bn \
FROM barra_factors \
WHERE indname1 in {tuple(SECTORS['FINANCIAL'])} \
AND nonestu = 0 \
GROUP BY indname1 \
ORDER BY avg_leverage DESC;"
sub_sector_avg_lev_marketcap_results = pd.read_sql(sub_sector_avg_lev_marketcap_query, engine)
print(f'Average leverage by financial sub-sector:\n {sub_sector_avg_lev_marketcap_results}')


In [ ]:
df = sub_sector_avg_lev_marketcap_results
fig = px.scatter(
    df,
    x='avg_leverage',
    y='marketcap_bn',
    hover_name='indname1',
    text='indname1',
    size_max=60,
    title='Average Leverage vs Market Cap for Financial Sub-Sectors (2007)',
    labels={
        'avg_leverage': 'Average Leverage',
        'marketcap_bn': 'Market Cap (Billions)'
    },
)

fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_dark', height=700)

fig.show()

### ANALYSIS 3.2: Industry Leverage Comparison
What we see is there isn't a correlation between leverage of sectors and their market cap. <br>
However, we see that the security assets have the largest market cap among the financial sector and are leveraged in line with the market as a whole. Similarly, Banks are in line with the market average but tend to have a smaller capitalization. What stands out is THRIFTS, who focus on taking deposits and converting them into securitized products like Mortgage Backed Securities, and Financial Services, who do not use a depositary model to fund the credit they extend are leveraged 1 std deviation higher than the market average, placing them in the top ~20% percentile compared to the market.  

4. **Index Membership**
   ### Question 4.1: Bank Index Distribution
   * Analyze distribution across SAP500, SAPVAL, SAPGRO
   * Calculate percentage of banks in each index
   * Compare with other sectors
   * Given that banks crashed in 2008, which index is most impacted?

In [ ]:
boolean_fields = [col for col, dtype in dtype_dict.items() if dtype == Boolean]
boolean_fields

In [ ]:
# check membership in sap500, sapval, sapgro
bank_percentage_in_indx_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS total_count,
    SUM(CASE WHEN sap500 = True THEN 1 ELSE 0 END) AS count_in_sap500,
    ROUND(SUM(CASE WHEN sap500 = True THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_in_sap500,

    SUM(CASE WHEN sapval = True THEN 1 ELSE 0 END) AS count_in_sapval,
    ROUND(SUM(CASE WHEN sapval = True THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_in_sapval,

    SUM(CASE WHEN sapgro = True THEN 1 ELSE 0 END) AS count_in_sapgro,
    ROUND(SUM(CASE WHEN sapgro = True THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_in_sapgro
FROM barra_factors
WHERE indname1 IN ('BANKS')
AND nonestu = 0
GROUP BY indname1;
'''
bank_percentage_in_indx_results = pd.read_sql(bank_percentage_in_indx_query, engine)
bank_percentage_in_indx_results

In [ ]:
# membership of other sectors not including banks
other_sectors_percentage_in_sapgro_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS total_count,
    
    SUM(CASE WHEN sapgro = True THEN 1 ELSE 0 END) AS count_in_sapgro,
    ROUND(SUM(CASE WHEN sapgro = True THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_in_sapgro
FROM barra_factors
WHERE indname1 NOT IN ('BANKS')
AND nonestu = 0
GROUP BY indname1
ORDER BY pct_in_sapgro DESC
'''
other_sectors_percentage_in_sapgro_results = pd.read_sql(other_sectors_percentage_in_sapgro_query, engine)
other_sectors_percentage_in_sapgro_results.head(10)

In [ ]:
other_sectors_percentage_in_sap500_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS total_count,
    SUM(CASE WHEN sap500 = True THEN 1 ELSE 0 END) AS count,
    ROUND(SUM(CASE WHEN sap500 = True THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_in_sap500
FROM barra_factors
WHERE indname1 NOT IN ('BANKS')
AND nonestu = 0
GROUP BY indname1
ORDER BY pct_in_sap500 DESC;
'''
other_sectors_percentage_in_sap500_results = pd.read_sql(other_sectors_percentage_in_sap500_query, engine)
other_sectors_percentage_in_sap500_results.head(10)

In [ ]:
other_sectors_percentage_in_sapval_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS total_count,
    SUM(CASE WHEN sapval = True THEN 1 ELSE 0 END) AS count,
    ROUND(SUM(CASE WHEN sapval = True THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_in_sapval
FROM barra_factors
WHERE indname1 NOT IN ('BANKS')
AND nonestu = 0
GROUP BY indname1
ORDER BY pct_in_sapval DESC;
'''
other_sectors_percentage_in_sapval_results = pd.read_sql(other_sectors_percentage_in_sapval_query, engine)
other_sectors_percentage_in_sapval_results.head(10)

### ANALYSIS 4.1: Bank Index Distribution
From 2007, data we see bank membership in SAP500 and SAP Value at 31.58% 30.26%, respectively. SAP Growth had a bank membership of 5.26%. This would lead us to believe that SAP500 and SAPVAL had the most impact.

### Question 4.2: Value vs Growth Classification
   * Study SAPVAL vs SAPGRO membership patterns
   * Examine MIDVAL vs MIDGRO distribution
   * Analyze SCVAL vs SCGRO composition
   * E.g. Do companies in SAPVAL have logically a higher SIZE and VALUE and lower GROWTH values compared to companies in SCVAL?

In [ ]:
# look at membership of sectors in sapval vs sapgro
membership_comparison_sapval_vs_sapgro_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS total_count,
    SUM(CASE WHEN sapval = True AND sapgro = False THEN 1 ELSE 0 END) AS count_in_sapval,
    ROUND(SUM(CASE WHEN sapval = True AND sapgro = False THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_in_sapval,
    SUM(CASE WHEN sapgro = True AND sapval = False THEN 1 ELSE 0 END) AS count_in_sapgro,
    ROUND(SUM(CASE WHEN sapgro = True AND sapval = False THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_in_sapgro
FROM barra_factors
GROUP BY indname1
ORDER BY total_count DESC;
'''
membership_comparison_sapval_vs_sapgro_results = pd.read_sql(membership_comparison_sapval_vs_sapgro_query, engine)
membership_comparison_sapval_vs_sapgro_results.head(10)



In [ ]:
# look at visualization of membership, stacked percentage bar chart
df_plot = membership_comparison_sapval_vs_sapgro_results.melt(
    id_vars=['industry', 'total_count'],
    value_vars=['pct_in_sapval', 'pct_in_sapgro'],
    var_name='index',
    value_name='percentage'
)

fig = px.bar(
    df_plot,
    x='industry',
    y='percentage',
    color='index',
    title='Percentage of Stocks in Each Industry that are in S&P Value vs S&P Growth (2007)',
    labels={
        'industry': 'Industry',
        'percentage': 'Percentage of Stocks in Index',
        'index': 'Index Membership'
    },
    template='plotly_dark'
)

fig.update_layout(
    xaxis_title='Industry',
    yaxis_title='Percentage of Stocks in Index',
    legend_title='Index',
    xaxis_tickangle=-45,
    height=600
)
fig.show()

In [ ]:
kws= ['val', 'gro', 'cap']
result_map_object = map(lambda kw: search_kw(kw), kws)

matched_cols = list(result_map_object)
matched_cols

In [ ]:
membership_comparison_sapval_vs_sapgro_avg_size_val_growth = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS total_count,
    AVG(CASE WHEN sapval = True AND sapgro = False THEN size END) AS avg_size_in_sapval,
    AVG(CASE WHEN sapgro = True AND sapval = False THEN size END) AS avg_size_in_sapgro,

    AVG(CASE WHEN sapval = True AND sapgro = False THEN value END) AS avg_val_in_sapval,
    AVG(CASE WHEN sapgro = True AND sapval = False THEN value END) AS avg_val_in_sapgro,

    AVG(CASE WHEN sapval = True and sapgro = False THEN capitalization END)/1e9 AS avg_cap_bn_in_sapval,
    AVG(CASE WHEN sapgro = True and sapval = False THEN capitalization END)/1e9 AS avg_cap_bn_in_sapgro
FROM barra_factors
GROUP BY indname1
ORDER BY total_count DESC;
'''
membership_comparison_sapval_vs_sapgro_avg_size_val_growth_results = pd.read_sql(membership_comparison_sapval_vs_sapgro_avg_size_val_growth, engine)
membership_comparison_sapval_vs_sapgro_avg_size_val_growth_results.head(10)

In [ ]:
kws = ['midval', 'midgro']
result_map_object = map(lambda kw: search_kw(kw), kws)
matched_cols = list(result_map_object)
matched_cols

In [ ]:
# * Examine MIDVAL vs MIDGRO distribution
midval_midgro_comparison_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS total_count,
    SUM(CASE WHEN midval = True and midgro = False THEN 1 ELSE 0 END) AS count_midval,
    ROUND(SUM(CASE WHEN midval = True and midgro = False THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_midval,
    SUM(CASE WHEN midgro = True and midval = False THEN 1 ELSE 0 END) AS count_midgro,
    ROUND(SUM(CASE WHEN midgro = True and midval = False THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_midgro
FROM barra_factors
GROUP BY indname1
ORDER BY total_count DESC;
'''
midval_midgro_comparison_results = pd.read_sql(midval_midgro_comparison_query, engine)
midval_midgro_comparison_results.head(10)

In [ ]:
# look at visualization of membership, stacked percentage bar chart
df_plot = midval_midgro_comparison_results.melt(
    id_vars=['industry', 'total_count'],
    value_vars=['pct_midval', 'pct_midgro'],
    var_name='index',
    value_name='percentage'
)

fig = px.bar(
    df_plot,
    x='industry',
    y='percentage',
    color='index',
    title='Percentage of Stocks in Each Industry that are in MidValue vs MidGrowth (2007)',
    labels={
        'industry': 'Industry',
        'percentage': 'Percentage of Stocks in Index',
        'index': 'Index Membership'
    },
    template='plotly_dark'
)

fig.update_layout(
    xaxis_title='Industry',
    yaxis_title='Percentage of Stocks in Index',
    legend_title='Index',
    xaxis_tickangle=-45,
    height=600
)
fig.show()

In [ ]:
kws = ['scval', 'scgro']
result_map_object = map(lambda kw: search_kw(kw), kws)
matched_cols = list(result_map_object)
matched_cols

In [ ]:
# * Analyze SCVAL vs SCGRO composition
scval_scgro_composition_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS total_count,
    SUM(CASE WHEN scval = True and scgro = False THEN 1 ELSE 0 END) AS count_scval,
    ROUND(SUM(CASE WHEN scval = True and scgro = False THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_scval,
    SUM(CASE WHEN scgro = True and scval = False THEN 1 ELSE 0 END) AS count_scgro,
    ROUND(SUM(CASE WHEN scgro = True and scval = False THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_scgro
FROM barra_factors
GROUP BY indname1
ORDER BY total_count DESC;
'''
scval_scgro_composition_results = pd.read_sql(scval_scgro_composition_query, engine)
scval_scgro_composition_results.head(10)

In [ ]:
# look at visualization of membership, stacked percentage bar chart
df_plot = scval_scgro_composition_results.melt(
    id_vars=['industry', 'total_count'],
    value_vars=['pct_scval', 'pct_scgro'],
    var_name='index',
    value_name='percentage'
)

fig = px.bar(
    df_plot,
    x='industry',
    y='percentage',
    color='index',
    title='Percentage of Stocks in Each Industry that are in SCVAL vs SCGRO (2007)',
    labels={
        'industry': 'Industry',
        'percentage': 'Percentage of Stocks in Index',
        'index': 'Index Membership'
    },
    template='plotly_dark'
)

fig.update_layout(
    xaxis_title='Industry',
    yaxis_title='Percentage of Stocks in Index',
    legend_title='Index',
    xaxis_tickangle=-45,
    height=600
)
fig.show()

In [ ]:
#  Composition of sapval, sapgro, midval, midgro, scval, scgro and averages of value, growth, capitalization
 
idx_composition_val_query = '''
SELECT
    AVG(CASE WHEN sapval = True THEN value END) AS avg_val_in_sapval,
    AVG(CASE WHEN sapgro = True THEN value END) AS avg_val_in_sapgro,
    AVG(CASE WHEN midval = True THEN value END) AS avg_val_in_midval,
    AVG(CASE WHEN midgro = True THEN value END) AS avg_val_in_midgro,
    AVG(CASE WHEN scval = True THEN value END) AS avg_val_in_scval,
    AVG(CASE WHEN scgro = True THEN value END) AS avg_val_in_scgro
FROM barra_factors
WHERE nonestu = 0
'''
idx_composition_val_results = pd.read_sql(idx_composition_val_query, engine)
idx_composition_val_results

In [ ]:
# plot the average value factor for each index 
plot_df = idx_composition_val_results.transpose().reset_index()
plot_df.columns = ['index', 'avg_value']

fig = px.bar(
    plot_df,
    x='index',
    y='avg_value',
    title='Average Value Factor for Each Index (2007)',
    labels={
        'index': 'Index',
        'avg_value': 'Average Value Factor'
    },
    template='plotly_dark'
)
fig.update_layout(
    xaxis_title='Index',
    yaxis_title='Average Value Factor',
    legend_title='Index',
    xaxis_tickangle=-45,
    height=600
)

In [ ]:
idx_composition_growth_query = '''
SELECT
    AVG(CASE WHEN sapval = True THEN growth END) AS avg_growth_in_sapval,
    AVG(CASE WHEN sapgro = True THEN growth END) AS avg_growth_in_sapgro,
    AVG(CASE WHEN midval = True THEN growth END) AS avg_growth_in_midval,
    AVG(CASE WHEN midgro = True THEN growth END) AS avg_growth_in_midgro,
    AVG(CASE WHEN scval = True THEN growth END) AS avg_growth_in_scval,
    AVG(CASE WHEN scgro = True THEN growth END) AS avg_growth_in_scgro
FROM barra_factors
WHERE nonestu = 0
'''
idx_composition_growth_results = pd.read_sql(idx_composition_growth_query, engine)
idx_composition_growth_results

In [ ]:
# plot the average value factor for each index 
plot_df = idx_composition_growth_results.transpose().reset_index()
plot_df.columns = ['index', 'avg_growth']

fig = px.bar(
    plot_df,
    x='index',
    y='avg_growth',
    title='Average Growth Factor for Each Index (2007)',
    labels={
        'index': 'Index',
        'avg_growth': 'Average Growth Factor'
    },
    template='plotly_dark'
)
fig.update_layout(
    xaxis_title='Index',
    yaxis_title='Average Growth Factor',
    legend_title='Index',
    xaxis_tickangle=-45,
    height=600
)
fig.show()

In [ ]:
idx_composition_cap_query = '''
SELECT
    AVG(CASE WHEN sapval = True THEN capitalization END)/1e9 AS avg_cap_bn_in_sapval,
    AVG(CASE WHEN sapgro = True THEN capitalization END)/1e9 AS avg_cap_bn_in_sapgro,
    AVG(CASE WHEN midval = True THEN capitalization END)/1e9 AS avg_cap_bn_in_midval,
    AVG(CASE WHEN midgro = True THEN capitalization END)/1e9 AS avg_cap_bn_in_midgro,
    AVG(CASE WHEN scval = True THEN capitalization END)/1e9 AS avg_cap_bn_in_scval,
    AVG(CASE WHEN scgro = True THEN capitalization END)/1e9 AS avg_cap_bn_in_scgro
FROM barra_factors
WHERE nonestu = 0
'''
idx_composition_cap_results = pd.read_sql(idx_composition_cap_query, engine)
idx_composition_cap_results

In [ ]:
plot_df = idx_composition_cap_results.transpose().reset_index()
plot_df.columns = ['index', 'avg_cap_bn']

fig = px.bar(
    plot_df,
    x='index',
    y='avg_cap_bn',
    title='Average Market Capitalization (Billions) for Each Index (2007)',
    labels={
        'index': 'Index',
        'avg_cap_bn': 'Average Market Cap (Billions)'
    },
    template='plotly_dark'
)
fig.update_layout(
    xaxis_title='Index',
    yaxis_title='Average Market Cap (Billions)',
    legend_title='Index',
    xaxis_tickangle=-45,
    height=600
)
fig.show()

### ANALYSIS: 4.2: Value vs Growth Classification
I decided to analyze the stocks that were exclusively listed in each indice to see how membership is distributed across the market. If I hadn't we would've double counted stocks.<br>
For the index composition, I did not force this exclusivity because the membership (even if duplicate) is a reflection of the indices.

5. **Multi-Industry Analysis**
   ### Question 5.1: Industry Overlap
   * Identify firms with non-zero WGT2%
   * Compare diversified vs focused firms
   * Analyze impact on risk metrics


In [ ]:
kw = 'wgt'
search_result = search_kw(kw)
search_result

In [ ]:
kws = ['yield', 'earn', 'trade']
result_map_object = map(lambda kw: search_kw(kw), kws)
matched_cols = list(result_map_object)
matched_cols

In [ ]:
firms_with_nonzero_wgt2_pct_query = '''
SELECT
    name,
    indname1 AS industry,
    wgt2_pct,
    capitalization/1e9 AS marketcap_bn,
    growth,
    value,
    leverage,
    voltilty,
    yield,
    earnyld,
    earnvar,
    tradeact
FROM barra_factors
WHERE wgt2_pct > 0
AND nonestu = 0
ORDER BY wgt2_pct DESC;
'''
firms_with_nonzero_wgt2_pct_results = pd.read_sql(firms_with_nonzero_wgt2_pct_query, engine)
firms_with_nonzero_wgt2_pct_results.head(10)

In [ ]:
firms_with_zero_wgt2_pct_query = '''
SELECT
    name,
    indname1 AS industry,
    wgt2_pct,
    capitalization/1e9 AS marketcap_bn,
    growth,
    value,
    leverage,
    voltilty,
    yield,
    earnyld,
    earnvar,
    tradeact
FROM barra_factors
WHERE wgt2_pct = 0
AND nonestu = 0
ORDER BY name;
'''
firms_with_zero_wgt2_pct_results = pd.read_sql(firms_with_zero_wgt2_pct_query, engine)
firms_with_zero_wgt2_pct_results.head(10)

In [ ]:
diversified_vs_focused_query = '''
SELECT
    CASE
        WHEN wgt2_pct > 0 THEN 'Diversified'
        ELSE 'Focused'
    END AS firm_type,
    COUNT(*) AS count,
    AVG(growth) AS avg_growth,
    AVG(value) AS avg_value,
    AVG(leverage) AS avg_leverage,
    AVG(voltilty) AS avg_volatility,
    AVG(yield) AS avg_yield,
    AVG(earnyld) AS avg_earnyld,
    AVG(earnvar) AS avg_earnvar,
    AVG(tradeact) AS avg_tradeact
FROM barra_factors
WHERE nonestu = 0
GROUP BY firm_type;
'''
diversified_vs_focused_results = pd.read_sql(diversified_vs_focused_query, engine)
diversified_vs_focused_results

In [ ]:
df_melted = diversified_vs_focused_results.melt(
    id_vars=['firm_type', 'count'],
    value_vars=['avg_growth', 'avg_value', 'avg_leverage', 'avg_volatility', 'avg_yield', 'avg_earnyld', 'avg_earnvar', 'avg_tradeact'],
    var_name='metric',
    value_name='average_value'
)

fig = px.bar(
    df_melted,
    x='metric',
    y='average_value',
    color='firm_type',
    title='Comparison of Diversified vs Focused Firms on Key Metrics (2007)',
    labels={
        'metric': 'Metric',
        'average_value': 'Average Value',
        'firm_type': 'Firm Type'
    },
    template='plotly_dark'
)
fig.update_layout(
    xaxis_title='Metric',
    yaxis_title='Average Value',
    legend_title='Firm Type',
    xaxis_tickangle=-45,
    height=600
)
fig.show()


   ### Question 5.2: Yield Analysis
   * Find highest-yield financial institutions
   * Study yield-risk relationships
   * Examine yield stability across size groups

In [ ]:
kws = ['yield', 'vol', 'earn', 'lev']
result_map_object = map(lambda kw: search_kw(kw), kws)
matched_cols = list(result_map_object)
matched_cols

In [ ]:
high_yield_financial_query = f'''
SELECT
    name,
    indname1 AS industry,
    yield,
    voltilty,
    earnyld,
    earnvar,
    leverage,
    capitalization/1e9 AS marketcap_bn
FROM barra_factors
WHERE indname1 IN {tuple(SECTORS['FINANCIAL'])}
AND nonestu = 0
ORDER BY yield DESC;
'''
high_yield_financial_results = pd.read_sql(high_yield_financial_query, engine)
high_yield_financial_results.head(10)

In [ ]:
high_yield_financial_results.describe()

In [ ]:
# plot yield distribution among financial firms
fig = px.histogram(
    high_yield_financial_results,
    x='earnyld',
    nbins=50,
    title='Distribution of Earnings Yield among Financial Firms (2007)',
    labels={
        'earnyld': 'Earnings Yield'
    },
    marginal='box',
    template='plotly_dark'
)
fig.update_layout(
    xaxis_title='Earnings Yield',
    yaxis_title='Count of Firms',
    height=600
)
fig.show()

In [ ]:
# # compare leverage to volality and show the size of the stock by market cap
# df = top_ten_banks_by_lev_results

# fig = px.scatter(
#     df,
#     x='leverage',
#     y='voltilty',
#     size='marketcap_bn',
#     color='earnings_yield',
#     hover_name='name',
#     text='name',
#     size_max=60,
#     title='Leverage vs Volatility for Top 10 Leveraged Banks (2007)',
#     labels={
#         'leverage': 'Leverage',
#         'voltilty': 'Volatility',
#         'marketcap_bn': 'Market Cap (Billions)',
#         'earnings_yield': 'Earnings Yield'
#     },
# )


# # clean up label and positioning and style
# fig.update_traces(textposition='top center')
# fig.update_layout(
#     template='plotly_dark',
#     height=700,
#     xaxis=dict(showgrid=True, zeroline=False),
#     yaxis=dict(showgrid=True, zeroline=False)
# )
# fig.show()

# compare leverage to volatility and show the size of the stock by market cap for firms with earning yield above 1 std deviation above the mean
high_yield_financial_results_filtered = high_yield_financial_results[high_yield_financial_results['earnyld'] > 1]
fig = px.scatter(
    high_yield_financial_results_filtered,
    x='leverage',
    y='voltilty',
    size='marketcap_bn',
    color='earnyld',
    hover_name='name',
    size_max=60,
    title='Leverage vs Volatility for High Yield Financial Firms (2007)',
    labels={
        'leverage': 'Leverage',
        'voltilty': 'Volatility',
        'marketcap_bn': 'Market Cap (Billions)',
        'earnyld': 'Earnings Yield'
    },
)
fig.update_traces(textposition='top center')
fig.update_layout(
    template='plotly_dark',
    height=700,
    xaxis=dict(showgrid=True, zeroline=False),
    yaxis=dict(showgrid=True, zeroline=False)
)
fig.show()

# CTEs, Views, and Subqueries: Three Ways to Organize SQL Logic

## Overview
SQL offers three powerful techniques for organizing complex queries and reusing calculations: CTEs (Common Table Expressions), Views, and Subqueries. Each serves different purposes and has distinct advantages.

## 1. Common Table Expressions (CTEs) - WITH Clause

### What is a CTE?
A CTE creates a temporary named result set that exists only for the duration of a single query. Think of it as a variable that holds query results temporarily.

### Syntax
```sql
WITH temp_name AS (
    -- Your query here
    SELECT ...
)
SELECT * FROM temp_name;
```

### When to Use CTEs
- **Complex multi-step calculations** within a single query
- **Need to reference the same result multiple times** in one query
- **Want readable, self-documenting code** with named intermediate steps
- **Recursive operations** (like hierarchical data traversal)

### Example
```sql
WITH momentum_stats AS (
    SELECT AVG(MOMENTUM) as mean, STDDEV(MOMENTUM) as stddev
    FROM barra_factors WHERE NONESTU = 0
)
SELECT NAME, MOMENTUM,
       (MOMENTUM - mean) / stddev as z_score
FROM barra_factors, momentum_stats
WHERE MOMENTUM > mean + stddev;
```

## 2. Views - Persistent Named Queries

### What is a View?
A View is a stored query definition that acts like a virtual table. Unlike CTEs, views persist in the database and can be used across multiple queries and sessions.

### Syntax
```sql
CREATE VIEW view_name AS
SELECT ...;

-- Use it like a table
SELECT * FROM view_name;
```

### When to Use Views
- **Frequently used complex queries** that multiple users/queries need
- **Simplifying database access** for non-technical users
- **Security** - restrict access to specific columns/rows
- **Consistent business logic** across applications

### Example
```sql
CREATE VIEW high_risk_companies AS
SELECT * FROM barra_factors
WHERE LEVERAGE > 3 AND VOLTILTY > 50;

-- Anyone can now easily query high-risk companies
SELECT * FROM high_risk_companies WHERE INDNAME1 = 'BANKS';
```

## 3. Subqueries - Inline Nested Queries

### What is a Subquery?
A subquery is a query nested inside another query. It runs first and provides results to the outer query.

### Syntax
```sql
-- In WHERE clause
SELECT * FROM table1
WHERE column > (SELECT AVG(column) FROM table2);

-- In FROM clause
SELECT * FROM (SELECT ... FROM ...) AS derived_table;
```

### When to Use Subqueries
- **Simple one-time calculations** that don't need naming
- **Filtering based on aggregate values** (WHERE clause)
- **Quick comparisons** against calculated values
- **When you only need the result once**

### Example
```sql
-- Find companies with above-average momentum
SELECT NAME, MOMENTUM
FROM barra_factors
WHERE MOMENTUM > (
    SELECT AVG(MOMENTUM)
    FROM barra_factors
    WHERE NONESTU = 0
);
```

## Comparison Table

| Feature | CTE | View | Subquery |
|---------|-----|------|----------|
| **Scope** | Single query | Database-wide | Single query |
| **Persistence** | Temporary | Permanent (until dropped) | Temporary |
| **Reusability** | Within same query | Across all queries | Not reusable |
| **Performance** | Calculated once | Pre-optimized | Calculated each use |
| **Readability** | Excellent | N/A (hidden) | Can be complex |
| **Best for** | Multi-step logic | Common queries | Simple filters |

## Decision Guide

```sql
-- Use CTE when you need intermediate results multiple times in ONE query
WITH industry_avg AS (
    SELECT INDNAME1, AVG(LEVERAGE) as avg_lev
    FROM barra_factors GROUP BY INDNAME1
)
SELECT * FROM barra_factors b
JOIN industry_avg i ON b.INDNAME1 = i.INDNAME1
WHERE b.LEVERAGE > i.avg_lev * 2;

-- Use View when the same logic is needed ACROSS MULTIPLE queries
CREATE VIEW market_statistics AS
SELECT AVG(LEVERAGE) as avg_lev, AVG(MOMENTUM) as avg_mom
FROM barra_factors WHERE NONESTU = 0;

-- Use Subquery for SIMPLE, ONE-TIME calculations
SELECT * FROM barra_factors
WHERE LEVERAGE > (SELECT AVG(LEVERAGE) * 2 FROM barra_factors);
```

## Real-World Analogy
- **CTE**: Like calculating subtotals on scratch paper while solving a math problem
- **View**: Like creating a summary report that many people will reference
- **Subquery**: Like doing a quick mental calculation to make a decision

The key is choosing the right tool for your specific need: CTEs for complex single queries, Views for reusable logic, and Subqueries for simple inline calculations.

# Understanding Database Indexes in SQL

## What is an Index?
Think of an index in a database like an index in a book:
- A book index helps you find pages without reading the whole book
- A database index helps find data without scanning the whole table
- Both trade storage space for faster lookup speed

## Types of Indexes

### 1. Single-Column Index
```sql
CREATE INDEX idx_barrid ON barra_factors(BARRID);
```
- Like looking up words in a book's index
- Best for finding exact matches
- Example: Finding a specific stock by its BARRID

### 2. Composite Index (Multiple Columns)
```sql
CREATE INDEX idx_ind_cap ON barra_factors(INDNAME1, CAPITALIZATION);
```
- Like a book indexed by both topic and subtopic
- Useful for queries that filter on multiple columns
- Example: Finding banks with market cap > $1B

## When Indexes Help

### Good Use Cases:
1. **WHERE Clauses**
   ```sql
   WHERE BARRID = 'ABC123'  -- Very fast with index
   ```

2. **JOIN Operations**
   ```sql
   JOIN another_table ON barra_factors.BARRID = another_table.BARRID
   ```

3. **ORDER BY**
   ```sql
   ORDER BY CAPITALIZATION DESC  -- Faster with index
   ```

### When Indexes Don't Help:
1. **Function on Column**
   ```sql
   WHERE LOWER(INDNAME1) = 'banks'  -- Index not used
   ```

2. **Leading Wildcard**
   ```sql
   WHERE BARRID LIKE '%123'  -- Index not used
   ```

## Performance Trade-offs

### Advantages:
- Faster SELECT queries
- Faster WHERE clause evaluation
- Improved sorting performance

### Disadvantages:
- Extra disk space needed
- Slower INSERT and UPDATE operations
- Maintenance overhead

## Real-World Analogy
Think of a library:
- Books without index: Must scan every page to find content
- Books with index: Jump directly to relevant pages
- Card catalog: Like a database index for all books


# EXPLAIN Plans
- Understanding query execution plans
- EXPLAIN shows how SQLite will execute your query (which indexes, scan types, etc.)
- Helps identify performance bottlenecks and missing indexes
- Different databases have different EXPLAIN syntax (EXPLAIN ANALYZE in PostgreSQL)



# Database Transactions: Ensuring Data Integrity

## What is a Transaction?
A transaction is a sequence of database operations that are treated as a single unit of work. Either ALL operations succeed, or NONE of them do. This "all-or-nothing" approach ensures your database never ends up in an inconsistent state.

## The ACID Properties
Transactions guarantee four critical properties:

### **A**tomicity
- All operations in a transaction succeed or all fail
- No partial updates allowed
- Like a bank transfer: money leaves one account AND enters another, or neither happens

### **C**onsistency
- Database remains in a valid state before and after the transaction
- All rules and constraints are maintained
- Prevents corruption of data relationships

### **I**solation
- Concurrent transactions don't interfere with each other
- Each transaction operates as if it's the only one running
- Prevents "dirty reads" of uncommitted data

### **D**urability
- Once committed, changes persist even if system crashes
- Data is written to permanent storage
- Survived power outages and system failures

## Basic Transaction Syntax

```sql
-- Start a transaction
BEGIN TRANSACTION;  -- or just BEGIN

-- Your SQL operations here
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;

-- End the transaction (choose one):
COMMIT;    -- Save all changes permanently
ROLLBACK;  -- Undo all changes since BEGIN
```

## Real-World Example: Bank Transfer

```sql
-- Scenario: Transfer $1000 from Alice to Bob
BEGIN TRANSACTION;

-- Deduct from Alice
UPDATE accounts
SET balance = balance - 1000
WHERE customer_name = 'Alice';

-- Check if Alice had sufficient funds
SELECT balance FROM accounts WHERE customer_name = 'Alice';
-- If balance is negative, we should ROLLBACK

-- Add to Bob
UPDATE accounts
SET balance = balance + 1000
WHERE customer_name = 'Bob';

-- If everything looks good:
COMMIT;
-- If there's any problem:
-- ROLLBACK;
```

## Transaction in Your Barra Context

```sql
-- Safe way to adjust leverage for risk analysis
BEGIN TRANSACTION;

-- Save original state
CREATE TEMP TABLE leverage_backup AS
SELECT BARRID, LEVERAGE FROM barra_factors WHERE INDNAME1 = 'BANKS';

-- Simulate leverage increase
UPDATE barra_factors
SET LEVERAGE = LEVERAGE * 1.1
WHERE INDNAME1 = 'BANKS';

-- Analyze the impact
SELECT
    COUNT(*) as high_risk_count,
    AVG(LEVERAGE) as new_avg_leverage
FROM barra_factors
WHERE INDNAME1 = 'BANKS' AND LEVERAGE > 3;

-- Decide based on analysis:
-- ROLLBACK;  -- Undo the simulation
-- or
-- COMMIT;    -- Keep the changes
```

## SQLite Transaction Modes

```sql
-- SQLite supports three transaction modes:

-- 1. DEFERRED (default)
BEGIN DEFERRED TRANSACTION;
-- Acquires locks only when needed
-- Allows concurrent reads

-- 2. IMMEDIATE
BEGIN IMMEDIATE TRANSACTION;
-- Acquires write lock immediately
-- Prevents other writes but allows reads

-- 3. EXCLUSIVE
BEGIN EXCLUSIVE TRANSACTION;
-- Acquires exclusive lock
-- No other connections can read or write
```

## Common Use Cases

### 1. **Data Import**
```sql
BEGIN TRANSACTION;
-- Import thousands of records
INSERT INTO barra_factors VALUES (...);
INSERT INTO barra_factors VALUES (...);
-- ... many more inserts
COMMIT;  -- All succeed or all fail
```

### 2. **Cleanup Operations**
```sql
BEGIN TRANSACTION;
-- Delete related data from multiple tables
DELETE FROM order_items WHERE order_id = 123;
DELETE FROM orders WHERE id = 123;
DELETE FROM customer_history WHERE order_id = 123;
COMMIT;  -- Ensures referential integrity
```

### 3. **Complex Updates**
```sql
BEGIN TRANSACTION;
-- Rebalance portfolio
UPDATE holdings SET weight = weight * 0.9 WHERE sector = 'TECH';
UPDATE holdings SET weight = weight * 1.1 WHERE sector = 'FINANCE';
-- Verify weights still sum to 100%
SELECT SUM(weight) FROM holdings;
-- If correct: COMMIT;
-- If wrong: ROLLBACK;
```

## Automatic Transactions in SQLite

```sql
-- SQLite wraps individual statements in implicit transactions
INSERT INTO test VALUES (1);  -- Automatically wrapped in BEGIN...COMMIT

-- But for multiple statements, explicit is better:
BEGIN;
INSERT INTO test VALUES (1);
INSERT INTO test VALUES (2);
COMMIT;  -- Both inserts succeed or both fail
```

## Error Handling Pattern

```python
# Python example with error handling
import sqlite3

conn = sqlite3.connect('database.db')
try:
    conn.execute("BEGIN")
    conn.execute("UPDATE accounts SET balance = balance - 100 WHERE id = 1")
    conn.execute("UPDATE accounts SET balance = balance + 100 WHERE id = 2")
    
    # Check for business logic
    result = conn.execute("SELECT balance FROM accounts WHERE id = 1").fetchone()
    if result[0] < 0:
        raise ValueError("Insufficient funds")
    
    conn.commit()
    print("Transaction successful")
except Exception as e:
    conn.rollback()
    print(f"Transaction failed: {e}")
finally:
    conn.close()
```

## Alternative FASTER
```python
with Session() as session:
    # Prepare all data in memory first (Pandas is great for this)
    update_data = [
        {"new_val": row.val, "target_id": row.id} 
        for row in df.itertuples()
    ]
    
    # Send everything in one batch
    session.execute(
        text("UPDATE my_table SET column = :new_val WHERE id = :target_id"),
        update_data
    )
    
    # Final decision
    confirm = input("Batch update ready. Commit? ")
    if confirm == 'y':
        session.commit()
```


## Key Points to Remember

1. **Transactions are temporary** until committed
2. **ROLLBACK undoes everything** since BEGIN
3. **Nested transactions** aren't supported in SQLite
4. **Automatic rollback** occurs if connection closes without commit
5. **Write operations lock the database** in SQLite (it's single-writer)

The power of transactions is that they protect your data integrity even when things go wrong - network failures, power outages, or logical errors in your code. They ensure your database is always in a consistent, valid state!

# Understanding Primary Keys in SQL

## What is a Primary Key?
A primary key is a column (or set of columns) that:
- Uniquely identifies each row in a table
- Cannot contain NULL values
- Must have unique values
- Automatically creates an index
- Only one primary key per table

## Real World Analogy
Think of:
- Social Security Numbers for people
- ISBN numbers for books
- License plate numbers for cars
- Student ID numbers in a university

## Creating Primary Keys

### During Table Creation
```sql
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,  -- Single column primary key
    name TEXT,
    grade REAL
);
```

### Composite Primary Key (Multiple Columns)
```sql
CREATE TABLE class_enrollment (
    student_id INTEGER,
    course_id INTEGER,
    semester TEXT,
    PRIMARY KEY (student_id, course_id, semester)
);
```

## Benefits of Primary Keys

1. **Data Integrity**
   - Prevents duplicate records
   - Ensures each record can be uniquely identified
   - No NULL values allowed

2. **Referential Integrity**
   - Other tables can reference this unique identifier
   - Enables proper relationships between tables
   - Foundation for foreign keys

3. **Performance**
   - Automatically creates an index
   - Speeds up searches and joins
   - Optimizes database operations

## Primary Key vs. Index
| Feature | Primary Key | Index |
|---------|-------------|--------|
| Uniqueness | Required | Optional |
| NULL values | Not allowed | Allowed |
| Per table | Only one | Multiple allowed |
| Purpose | Data integrity | Performance |
| Auto-indexed | Yes | N/A |

## Best Practices
1. Choose meaningful primary keys when possible
2. Use simple and stable values
3. Avoid using data that might change
4. Consider using auto-incrementing numbers for simplicity
5. Keep the primary key as small as possible


# Understanding Performance: Primary Keys vs. Indices

## Key Performance Comparison
Based on the sample results:
```
Query time without index: 0.0038 seconds
Query time with index:    0.0004 seconds
Query time with PK:       0.0003 seconds
```

## Why PK is (Usually) Faster
1. **Automatic Clustering**
   - PKs often physically organize data
   - Database typically clusters data around PK
   - Direct access to disk location

2. **Unique Index**
   - PK automatically creates a unique index
   - Can stop searching after first match
   - No need to check for duplicates

## However, PKs Aren't Always Faster
PKs might be slower when:
1. **Composite Keys**
   ```sql
   PRIMARY KEY (BARRID, date)  -- Multiple columns
   ```
   vs.
   ```sql
   INDEX idx_barrid (BARRID)   -- Single column
   ```

2. **Large Primary Keys**
   - If PK is large (like a long string)
   - Takes more space in index
   - More I/O operations needed

3. **Non-Sequential Values**
   - Random strings (like BARRIDs)
   - vs. Auto-incrementing integers
   - Can cause page splits

## Best Practice
- Use PK for data integrity
- Add separate indices for performance if needed
- Monitor query performance
- Choose based on your specific use case



# Understanding SQL JOINs and Table Aliases

## What is a JOIN?
A JOIN combines rows from two or more tables based on a related column between them. Think of it like:
- Matching records in an Excel VLOOKUP
- Connecting puzzle pieces that fit together
- Merging two spreadsheets based on a common ID

## Table Aliases (a and b)
In our query:
```sql
FROM barra_factors a       -- 'a' is 2007 data
JOIN barra_2008_pk b      -- 'b' is 2008 data
```
- `a` is a nickname for the 2007 table (barra_factors)
- `b` is a nickname for the 2008 table (barra_2008_pk)
- Makes the code shorter and clearer
- Helps distinguish which table each column comes from

## Breaking Down the Query

```sql
SELECT
    a.BARRID,             -- BARRID from 2007 table
    a.NAME,               -- Company name from 2007 table
    a.INDNAME1,           -- Industry from 2007 table
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,    -- 2007 market cap
    ROUND(b.CAPITALIZATION/1e9, 2) as CAP_2008_B,    -- 2008 market cap
    ROUND((b.CAPITALIZATION - a.CAPITALIZATION)/a.CAPITALIZATION * 100, 2) as PCT_CHANGE  -- % change
```

## How the JOIN Works
```sql
FROM barra_factors a
JOIN barra_2008_pk b ON a.BARRID = b.BARRID
```
- Matches rows where BARRID is the same in both tables
- Like this:
```
2007 (a)    JOIN   2008 (b)
BARRID=ABC  ←→     BARRID=ABC
BARRID=XYZ  ←→     BARRID=XYZ
```

## Filters in the Query
```sql
WHERE a.NONESTU = 0    -- Only estimation universe stocks
    AND a.CAPITALIZATION > 1e9           -- Only billion+ companies
```

## Visual Example:
```
2007 Table (a)          2008 Table (b)           RESULT
BARRID  CAP             BARRID  CAP      →       BARRID  CAP_2007  CAP_2008
ABC     100             ABC     90        →       ABC     100       90
XYZ     200             XYZ     150       →       XYZ     200       150
DEF     300             MNO     175       →       DEF     300       [no match]
```


## Other JOINS

- RIGHT JOIN keeps all rows from the right table, matches from left
  * Not supported in SQLite, but available in other SQLs
- LEFT JOIN keeps all rows from the left table, matches from right
- FULL OUTER JOIN - shows companies in either year
  * Not supported in SQLite, but available in other SQLs

In [ ]:
Image(image_folder / 'JOINS-VENN.jpg', width=400)

## Non-survivors (LEFT JOIN)

# Finding Missing Companies: Understanding the SQL Query

## Purpose
This query finds companies that existed in 2007 but disappeared in 2008 (possibly due to bankruptcy, mergers, or delistings), focusing on the most leveraged companies.

## Query Breakdown

### 1. Select Statement
```sql
SELECT
    a.BARRID,                                   -- Company identifier
    a.NAME,                                     -- Company name
    a.INDNAME1,                                 -- Primary industry
    ROUND(a.CAPITALIZATION/1e9, 2) as CAP_2007_B,  -- Market cap in billions
    a.LEVERAGE as LEVERAGE_2007                 -- Leverage ratio
```

### 2. Table Join
```sql
FROM barra_factors a                           -- 2007 data (alias 'a')
LEFT JOIN barra_2008_pk b                      -- 2008 data (alias 'b')
    ON a.BARRID = b.BARRID                     -- Matching on company ID
```
- LEFT JOIN keeps ALL 2007 records
- Matches with 2008 where possible
- NULLs where no match exists

### 3. Filters
```sql
WHERE b.BARRID IS NULL                         -- No match in 2008
    AND a.NONESTU = 0                          -- In estimation universe
```
- `b.BARRID IS NULL` finds disappeared companies
- `NONESTU = 0` focuses on significant stocks

### 4. Sorting and Limit
```sql
ORDER BY LEVERAGE_2007 DESC                    -- Highest leverage first
LIMIT 20;                                      -- Show top 20
```

## Example Results
```
BARRID  NAME        INDNAME1  CAP_2007_B  LEVERAGE_2007
LEH     Lehman     BANKS     45.2        2.89
BSC     Bear       BANKS     12.1        2.45
...
```


## NULL Handling Notes:

- NULL Handling: Working with missing data
- NULL represents unknown or missing values
- NULL comparisons require IS NULL/IS NOT NULL (not = or !=)
- COALESCE replaces NULL with a default value

```python
query = '''
-- Different ways to handle NULL values
SELECT
    COUNT(*) as total_rows,
    COUNT(PCT_CHANGE) as non_null_PCT_CHANGE,
    COUNT(*) - COUNT(PCT_CHANGE) as null_PCT_CHANGE
FROM big_losers_view;
'''
pd.read_sql(query, engine)
```


```python
query = '''
-- COALESCE to replace NULL with default

SELECT *,
    COALESCE(PCT_CHANGE, -99.0) as COALESCE_PCT_CHANGE
FROM big_losers_view
'''
pd.read_sql(query, engine)
```

* IS NULL vs = NULL (important distinction)

```python
query = '''
-- WRONG

SELECT *
FROM big_losers_view
WHERE PCT_CHANGE = NULL
LIMIT 10;
'''
pd.read_sql(query, engine)
```

```python
query = '''
-- WRONG

SELECT *
FROM big_losers_view
WHERE PCT_CHANGE <> NULL
LIMIT 10;
'''
pd.read_sql(query, engine)
```

```
query = '''
-- RIGHT

SELECT *
FROM big_losers_view
WHERE PCT_CHANGE is NULL
LIMIT 10;
'''
pd.read_sql(query, engine)
```

#LIKE Operator
LIKE is used for pattern matching in SQL, allowing you to search for partial matches in text. It uses two special characters:
- `%` matches any sequence of characters
- `_` matches any single character

## Breaking Down Our Query
```sql
SELECT
    BARRID,
    NAME,
    INDNAME1,
    ROUND(CAPITALIZATION/1e9, 2) as CAP_B,
    LEVERAGE
FROM barra_factors
WHERE (NAME LIKE '%LEHMAN%' OR NAME LIKE '%LEH%')
    AND NONESTU = 0;
```

## Pattern Matching Examples

### 1. `NAME LIKE '%LEHMAN%'`
- Matches: "LEHMAN BROTHERS", "LEHMAN CORP", "NEW LEHMAN INC"
- The `%` on both sides means "find LEHMAN anywhere in the name"

### 2. `NAME LIKE '%LEH%'`
- Matches: "LEH", "LEHMAN", "LEHBROS"
- Broader search to catch abbreviations

## Common LIKE Patterns
```sql
LIKE 'ABC%'    -- Starts with ABC
LIKE '%XYZ'    -- Ends with XYZ
LIKE '%PQR%'   -- Contains PQR
LIKE 'A_C'     -- Three letters, A and C with any character between
```

## Using OR in the Search
```sql
WHERE (NAME LIKE '%LEHMAN%' OR NAME LIKE '%LEH%')
```
- Matches either pattern
- Parentheses group the conditions
- More flexible search


# Advanced Research Questions: The Global Financial Crisis Through Barra Data

## Core Focus: Crisis Risk Dynamics
For all queries, remember to only use `a.NONESTU = 0` (2007 estimation universe) to avoid survivorship bias.

## Research Questions

1. **Financial Sector Collapse**
   ```sql
   -- Example structure comparing financial firms
   SELECT
       a.NAME,
       a.INDNAME1,
       a.VOLTILTY as VOL_2007,
       b.VOLTILTY as VOL_2008,
       a.LEVERAGE as LEV_2007
   FROM barra_factors a
   LEFT JOIN barra_2008_pk b ON a.BARRID = b.BARRID
   WHERE a.INDNAME1 IN ('BANKS', 'THRIFTS', 'SECASSET')
   AND a.NONESTU = 0
   ORDER BY a.LEVERAGE DESC;
   ```
   * Did high leverage in 2007 predict bank failures?
   * Which banks had highest volatility increases?
   * Were there warning signs in risk metrics?

2. **Auto Industry Crisis**
   * How did Ford's risk metrics compare to other automakers?
   * Did MOTORVEH companies show similar volatility patterns?
   * Was the auto sector crisis visible in BETA changes?

3. **Government Intervention Analysis**
   ### Question 3.1: Bailout Recipients
   * Compare Fannie Mae, Freddie Mac, and AIG metrics
   * Track their LEVERAGE and VOLTILTY changes
   * Did their BETAs signal systemic importance?

   ### Question 3.2: Industry Impact
   * How did FINSVCS vs BANKS risk metrics differ?
   * Which sectors showed contagion effects?
   * Impact on real estate related sectors (EQTYREIT)?

4. **Size and Value Effects**
   ### Question 4.1: Large vs Small Banks
   * Did size help banks survive?
   * Compare risk metrics across size groups
   * VALUE scores of failed vs surviving banks

   ### Question 4.2: Cross-Industry Size Impact
   * Were larger firms more resilient across all sectors?
   * Did small caps (SC600) show different patterns?
   * Size factor evolution during crisis

5. **Real Economy Impact**
   ### Question 5.1: Construction and Housing
   * Track CONSTRUC and EQTYREIT sector metrics
   * Compare with pre-crisis levels
   * Links between real estate and financial firms

   ### Question 5.2: Consumer Sectors
   * How did consumer discretionary (CLOTHING, RETAIL) fare?
   * Impact on luxury vs basic goods sectors
   * Signs of consumer stress in sector metrics

## Special Focus Areas
1. **Bear Stearns and Lehman Timeline**
   * Risk metric progression before collapse
   * Industry-wide impact after each failure
   * Contagion effects on surviving firms

2. **Real Estate Connection**
   * Track REITs and construction firms
   * Mortgage-related companies performance
   * Cross-sector risk transmission

3. **Government Action Impact**
   * Risk metrics before/after key interventions
   * Sector-wide changes post-bailouts
   * Differences between aided and non-aided firms

If you need additional data to answer these questions, try getting from yfinance, checking that the name matches (not just the ticker).

## Hints for project:

* For example, you could have one table for 2007 Barra USE3 data, one table for 2008 Barra USE3 data, and another table that maps Barra Industries to Barra Sectors. Then you can demonstrate your SQL expertise by joining the tables together in a query to determine the GFC impact on financial sector from 2007 to 2008.
* The mapping for industry to sectors is found in PDF 83-84 of https://www.alacra.com/alacra/help/barra_handbook_US.pdf
* For simplicity, you can assume a stock is in INDNAME1 (and ignore the less significant INDNAME2 etc.)